## Web Scraping With BeautifulSoup And Selenium

### What is Web Scraping? 

Web scraping is like being a detective on the internet. It’s a way to collect information (like text, prices, or names) from websites by using a computer program. It is an automated process of extracting data from websites using python code instead of manually copying and pasting it.

#### BeautifulSoup

BeautifulSoup is a super simple Python tool that helps you pull out data from web pages.BeautifulSoup is purely a parser. It takes an HTML string and it makes it easy for you can find elements, extract text, and pull attributes. It has no idea what a browser is, it just reads text. 

Pros:
- This makes it very fast and lightweight, but it only works on content that already exists in the raw HTML source.

- Fun for beginners: You can explore websites and collect cool info.

- Useful: Get data for projects, like tracking toy prices or weather updates.

#### Selenium

Selenium is a browser controller. It launches a real Chrome or Firefox browser, loads the page, lets JavaScript execute fully, and then lets you interact with it like a human would, clicking buttons, filling forms, scrolling down. Once the page is fully loaded, you can grab the HTML and hand it off to BeautifulSoup for parsing.

The key question to ask yourself: Is the data I want visible in View Source?

Yes → use requests + BeautifulSoup (fast, simple)

No → use Selenium (because JavaScript renders it after load)

In [ ]:
# Install Dependencies 

# Core scraping tools
! pip install requests beautifulsoup4 selenium

# Parser for BeautifulSoup
! pip install lxml

# Auto-manages ChromeDriver for Selenium
! pip install webdriver-manager

# Data analysis
! pip install pandas matplotlib

print("Libraries ready to import!")

In [21]:
import time   # For adding delays between requests

# HTTP & HTML Parsing 
import requests # Makes HTTP GET requests (like a browser)
from bs4 import BeautifulSoup  # Parses HTML 

#  Browser Automation (Selenium)
from selenium import webdriver # It lets you launch and control the browsers like you are using chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Data Handling 
import pandas as pd
import matplotlib.pyplot as plt

print("All imports successful.")

All imports successful.


In [22]:
# Configure Chrome options (Launch an headless browser)
chrome_options = Options()
chrome_options.add_argument("--headless")          
chrome_options.add_argument("--no-sandbox")         
chrome_options.add_argument("--disable-dev-shm-usage")  
chrome_options.add_argument("--disable-gpu")       
chrome_options.add_argument("--window-size=1920,1080")  

# A realistic User-Agent prevents some anti-bot measures from triggering
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0.0.0 Safari/537.36"
)

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=chrome_options)

print("Browser launched successfully!")
print(f"Chrome version: {driver.capabilities['browserVersion']}")

Browser launched successfully!
Chrome version: 146.0.7680.155


In [23]:
TARGET_URL = "https://bitcointreasuries.net/private-companies"


driver.get(TARGET_URL)


wait = WebDriverWait(driver, 20)

wait.until(EC.presence_of_element_located((By.TAG_NAME, "tr")))

time.sleep(3)

print("Page loaded and data rendered!")
print(f"Current URL: {driver.current_url}")
print(f"Page title:  {driver.title}")

Page loaded and data rendered!
Current URL: https://bitcointreasuries.net/private-companies
Page title:  BitcoinTreasuries.NET - Private Companies with Bitcoin Holdings - BitcoinTreasuries


In [24]:
rendered_html = driver.page_source


soup_bt = BeautifulSoup(rendered_html, "lxml")

table = soup_bt.find("table")

if table:
    print("Table found!")
   
    rows = table.find_all("tr")
    print(f"Rows found: {len(rows)} (including header)")
else:
    print("No table found. The page may not have fully rendered.")
    print("Try increasing the sleep time in Step 3.2.")

Table found!
Rows found: 75 (including header)


In [25]:
treasury_data = []

if table:
    thead = table.find("thead")
    if thead:
        headers_row = thead.find_all("th")
        column_names = [h.get_text(strip=True) for h in headers_row]
        column_names.insert(1, "Country")
        print("Fixed columns:", column_names)
    
    tbody = table.find("tbody")
    if tbody:
        for row in tbody.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) < 2:
                continue
            cell_values = [c.get_text(separator=" ", strip=True) for c in cells]
            treasury_data.append(cell_values)

print(f"Extracted {len(treasury_data)} company rows")
if treasury_data:
    print("Sample row:", treasury_data[0])


Fixed columns: ['#', 'Country', 'Name', 'Bitcoin', 'In USD', '/21M']
Extracted 73 company rows
Sample row: ['1', '🇺🇸', 'Block.one', '₿ 164,000', '$11,597M', '0.781%']


In [26]:
# Build DataFrame with the corrected column names
df_treasury = pd.DataFrame(treasury_data, columns=column_names)

# Print ALL rows — no truncation
pd.set_option("display.max_rows", None)       
pd.set_option("display.max_columns", None)    
pd.set_option("display.width", None)          
pd.set_option("display.max_colwidth", None) 

print(df_treasury.to_string(index=False))

 # Country                                  Name   Bitcoin   In USD   /21M
 1      🇺🇸                             Block.one ₿ 164,000 $11,597M 0.781%
 2      🇻🇬               Tether Holdings Limited  ₿ 96,184  $6,802M 0.458%
 3      🇺🇸            Stone Ridge Holdings Group  ₿ 10,000    $707M 0.048%
 4      🇺🇸                            SpaceX 🥬 🙌   ₿ 8,285    $586M 0.039%
 5      🇨🇭                  The Tezos Foundation   ₿ 2,903    $205M 0.014%
 6      🇺🇸                    Ionic Digital Inc.   ₿ 2,662    $188M 0.013%
 7      🇺🇸           Zap Solutions, Inc (Strike)   ₿ 1,500    $106M 0.007%
 8      🇺🇸                             GIGA Inc.   ₿ 1,252     $89M 0.006%
 9      🇫🇷                      Melanion Digital     ₿ 342     $24M 0.002%
10      🇺🇸                        Steak 'n Shake     ₿ 162     $11M 0.001%
11      🇧🇲                Meanwhile Incorporated     ₿ 123      $9M 0.001%
12      🇩🇪                         Evertz Pharma     ₿ 100      $7M 0.000%
13      🇦🇷               

In [27]:
# Build the DataFrame with the corrected 6 columns
column_names = ['#', 'Country', 'Name', 'Bitcoin', 'In USD', '/21M']

df_treasury = pd.DataFrame(treasury_data, columns=column_names)

print(f"DataFrame shape: {df_treasury.shape}")
df_treasury

DataFrame shape: (73, 6)


,#,Country,Name,Bitcoin,In USD,/21M
0,1,🇺🇸,Block.one,"₿ 164,000","$11,597M",0.781%
1,2,🇻🇬,Tether Holdings Limited,"₿ 96,184","$6,802M",0.458%
2,3,🇺🇸,Stone Ridge Holdings Group,"₿ 10,000",$707M,0.048%
3,4,🇺🇸,SpaceX 🥬 🙌,"₿ 8,285",$586M,0.039%
4,5,🇨🇭,The Tezos Foundation,"₿ 2,903",$205M,0.014%
5,6,🇺🇸,Ionic Digital Inc.,"₿ 2,662",$188M,0.013%
6,7,🇺🇸,"Zap Solutions, Inc (Strike)","₿ 1,500",$106M,0.007%
7,8,🇺🇸,GIGA Inc.,"₿ 1,252",$89M,0.006%
8,9,🇫🇷,Melanion Digital,₿ 342,$24M,0.002%
9,10,🇺🇸,Steak 'n Shake,₿ 162,$11M,0.001%


In [28]:
# Save the raw scraped data before any cleaning
df_treasury.to_csv("bitcoin_treasuries_raw.csv", index=False)
print("Saved: bitcoin_treasuries_raw.csv")

Saved: bitcoin_treasuries_raw.csv


In [29]:
# Always run this immediately after you're done scraping
driver.quit()
print("Browser closed.")

Browser closed.


In [30]:
URL = "https://www.coingecko.com"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

response = requests.get(URL, headers=headers, timeout=15)

print(f"Status code: {response.status_code}")
print(f"Page size:   {len(response.text):,} characters")

Status code: 200
Page size:   1,642,765 characters


In [31]:
soup = BeautifulSoup(response.text, "lxml")

print("Page title:", soup.title.text.strip())

table = soup.find("table")
if table:
    print("Table found!")
    rows = table.find_all("tr")
    print(f"Rows found: {len(rows)}")
else:
    print("No table found.")

Page title: Cryptocurrency Prices, Charts, and Crypto Market Cap | CoinGecko
Table found!
Rows found: 101


In [36]:
coin_data = []

tbody = table.find("tbody") if table else None

if tbody:
    for row in tbody.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) < 8:
            continue

        # Each cell's position maps to a column
        rank        = cells[1].get_text(strip=True)
        
        # Name and ticker are together in one cell — split them
        name_cell   = cells[2].get_text(separator="|", strip=True).split("|")
        name        = name_cell[0] if name_cell else ""
        ticker      = name_cell[1] if len(name_cell) > 1 else ""
        
        price       = cells[3].get_text(strip=True)
        change_1h   = cells[4].get_text(strip=True)
        change_24h  = cells[5].get_text(strip=True)
        change_7d   = cells[6].get_text(strip=True)
        volume_24h  = cells[7].get_text(strip=True)
        market_cap  = cells[8].get_text(strip=True) if len(cells) > 8 else ""

        coin_data.append({
            "Rank":       rank,
            "Name":       name,
            "Ticker":     ticker,
            "Price":      price,
            "1h %":       change_1h,
            "24h %":      change_24h,
            "7d %":       change_7d,
            "Volume 24h": volume_24h,
            "Market Cap": market_cap
        })

df_prices = pd.DataFrame(coin_data)

print(f"Scraped {len(df_prices)} coins")
df_prices

Scraped 100 coins


,Rank,Name,Ticker,Price,1h %,24h %,7d %,Volume 24h,Market Cap
0,1,Bitcoin,BTC,Buy,"$71,041.36",0.1%,2.4%,0.8%,8.7%
1,2,Ethereum,ETH,Buy,"$2,170.55",0.1%,2.6%,1.5%,16.5%
2,3,Tether,USDT,,$0.9996,0.0%,0.0%,0.0%,0.0%
3,4,BNB,BNB,Buy,$645.82,0.2%,2.6%,0.5%,7.1%
4,5,XRP,XRP,Buy,$1.42,0.1%,2.0%,2.8%,3.6%
5,6,USDC,USDC,,$0.9998,0.0%,0.0%,0.0%,0.0%
6,7,Solana,SOL,Buy,$92.03,0.0%,3.2%,2.3%,18.3%
7,8,TRON,TRX,Buy,$0.3131,0.0%,0.9%,3.4%,11.1%
8,9,Figure Heloc,FIGR_HELOC,Buy,$1.03,0.0%,0.1%,0.0%,1.5%
9,10,Dogecoin,DOGE,Buy,$0.09634,0.0%,4.0%,2.6%,2.0%


In [34]:
df_prices.to_csv("coingecko_prices_raw.csv", index=False)
print("Saved: coingecko_prices_raw.csv")

Saved: coingecko_prices_raw.csv
